In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Step 1: నిర్మాణాత్మక అవుట్పుట్‌ల కొరకు Pydantic మోడళ్లు నిర్వచించండి

ఈ మోడళ్లు ఏజెంట్లు తిరిగి ఇవ్వబోయే **స్కీమా**ని నిర్వచిస్తాయి. Pydanticతో `response_format` ఉపయోగించడం ద్వారా:
- ✅ రకం-భద్రత గల డేటా సంకలనం
- ✅ ఆటోమేటిక్ నిర్థారణ
- ✅ ఫ్రీ-టెక్స్ట్ ప్రతిస్పందనల నుండి పార్సింగ్ లోపాలు రాకపోవడం
- ✅ ఫీల్డుల ఆధారంగా సులభ conditional routing


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: హోటల్ బుకింగ్ టూల్ సృష్టించండి

ఈ టూల్ **availability_agent** గదులు లభ്യമున్నాయా అని తనిఖీ చేయడానికి పిలుస్తుంది. మేము `@ai_function` డెకొరేటర్ ఉపయోగిస్తాము:
- पाइథాన్ ఫంక్షన్‌ను AI-పిలవదగిన టూల్‌గా మార్చడం కోసం
- LLM కోసం JSON స్కీమాను ఆటోమేటిక్‌గా సృష్టించడానికి
- పారామితుల ధ్రువీకరణను నిర్వహించడానికి
- ఏజెంట్ల ద్వారా ఆటోమేటిక్ పిలుపులను అవకాశం కల్పించడానికి

ఈ డెమో కోసం:
- **స్టాక్‌హోమ్, సీయాటిల్, టోక్యో, లండన్, ఆంస్టర్డామ్** → గదులు ఉన్నాయి ✅
- **ఇతర అన్ని పట్టణాలు** → గదులు లేవు ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Step 3: రౌటింగ్ కోసం స్థితి ఫంక్షన్లను నిర్వచించండి

ఈ ఫంక్షన్లు ఏజెంట్ యొక్క ప్రతిస్పందనను పరిశీలించి వర్క్‌ఫ్లోలో ఏ పథాన్ని తీసుకోవాలో నిర్ణయిస్తాయి.

**ప్రధాన నమూనా:**
1. సందేశం `AgentExecutorResponse` కనుకున్నదో లేదో తనిఖీ చేయండి
2. నిర్మిత అవుట్పుట్ (Pydantic మోడల్) ని పార్స్ చేయండి
3. రౌటింగ్‌ను నియంత్రించడానికి `True` లేదా `False` ను తిరిగి ఇవ్వండి

వర్క్‌ఫ్లో ఈ పరిస్థితులను **ఎడ్జ్‌లపై** మూల్యాంకనం చేసి, ఎటువంటి నెక్స్ట్ ఎగ్జిక్యూటర్‌ను ఆహ్వానించాలో నిర్ణయిస్తుంది.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Step 4: కస్టమ్ డిస్ప్లే ఎగ్జిక్యూటర్ సృష్టించడం

ఎగ్జిక్యూటర్లు ట్రాన్స్ఫర్మేషన్లు లేదా సైడ్ ఎఫెక్ట్స్ నిర్వహించేది వర్క్ఫ్లో భాగాలు. మనం `@executor` డెకరేటర్ ఉపయోగించి చివరి ఫలితాన్ని ప్రదర్శించే కస్టమ్ ఎగ్జిక్యూటర్‌ను సృష్టిస్తాము.

**ప్రధాన భావనలు:**
- `@executor(id="...")` - ఒక ఫంక్షన్‌ను వర్క్ఫ్లో ఎగ్జిక్యూటర్‌గా నమోదు చేయడం
- `WorkflowContext[Never, str]` - ఇన్‌పుట్/ఔట్‌పుట్ కోసం టైపు సూచనలు
- `ctx.yield_output(...)` - చివరి వర్క్ఫ్లో ఫలితాన్ని ఉత్పత్తి చేస్తుంది


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Step 5: వాతావరణ చరాలను లోడ్ చేయండి

LLM క్లయింట్‌ను కాన్ఫిగర్ చేయండి. ఈ ఉదాహరణ క్రింది వాటితో పనిచేస్తుంది:
- **GitHub మోడల్స్** (GitHub టోకెన్‌తో ఉచిత స్థాయి)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Step 6: నిర్మాణం చేయబడిన అవుట్పుట్‌లతో AI ఏజెంట్లను సృష్టించండి

మేము **మూడు ప్రత్యేక ఏజెంట్లను** సృష్టించాము, ప్రతి ఒక్కటీ ఒక `AgentExecutor` లో అమర్చబడి ఉంది:

1. **availability_agent** - సాధనాన్ని ఉపయోగించి హోటల్ లభ్యతను తనిఖీ చేస్తుంది
2. **alternative_agent** - ప్రత్యామ్నాయ నగరాలను సూచిస్తుంది (గదులు లేని సమయంలో)
3. **booking_agent** - బుకింగ్ చేయమని ప్రేరేపిస్తుంది (గదులు లభ్యమైనప్పుడు)

**ప్రధాన లక్షణాలు:**
- `tools=[hotel_booking]` - ఏజెంట్ కు సాధనాన్ని అందిస్తుంది
- `response_format=PydanticModel` - నిర్మాణం చేయబడిన JSON అవుట్పుట్‌ను బలవంతం చేస్తుంది
- `AgentExecutor(..., id="...")` - వర్క్‌ఫ్లోలో ఉపయోగం కోసం ఏజెంట్ ను చుట్టి ఉంచుతుంది


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 7: షరతు ఆధారిత ఎడ్జెస్‌తో వర్క్‌ఫ్లోని నిర్మించడం

ఇప్పుడు `WorkflowBuilder` ను ఉపయోగించి షరతు మార్గదర్శకంతో గ్రాఫ్‌ను నిర్మిస్తాము:

**వర్క్‌ఫ్లో రూపొందింపు:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**ప్రధాన పద్ధతులు:**
- `.set_start_executor(...)` - ప్రవేశ బిందువును సెట్ చేస్తుంది
- `.add_edge(from, to, condition=...)` - షరతు ఆధారిత ఎడ్జ్‌ను చేర్చడము
- `.build()` - వర్క్‌ఫ్లోని తుది రూపంలో తయారు చేస్తుంది


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 8: పరీక్షా కేసు 1 ని నడపండి - నగరం అందుబాటులో లేదు (పారిస్)

మన సిమ్యులేషన్‌లో గదులు లేని పారిస్‌లో హోటల్స్‌ని కోరడం ద్వారా **అందుబాటు లేదు** మార్గాన్ని పరీక్షిద్దాం.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: టెస్ట్ కేసు 2 నడపండి - అందుబాటుతో నగరం (స్టాక్‌హోమ్)

ఇప్పుడు మనం స్టాక్‌হోమ్‌లో హోటల్స్ (మా సిమ్యులేషన్‌లో గదులు ఉన్నవి) కోసం **అందుబాటు** మార్గాన్ని పరీక్షしましょう.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ముఖ్యమైన విషయాలు మరియు తదుపరి దశలు

### ✅ మీరు నేర్చుకున్నవి:

1. **WorkflowBuilder నమూనా**
   - ప్రవేశ బిందువు నిర్వచించడానికి `.set_start_executor()` ఉపయోగించండి
   - షరతు ఆధారిత మార్గాన్ని సృష్టించడానికి `.add_edge(from, to, condition=...)` ఉపయోగించండి
   - పని ప్రవాహాన్ని నిలిపివేయడానికి `.build()`ని పిలవండి

2. **షరతు ఆధారిత మార్గం**
   - షరతు ఫంక్షన్లు `AgentExecutorResponse`ని పరిశీలిస్తాయి
   - మార్గ నిర్ణయాలు తీసుకోవడానికి నిర్మిత అవుట్పుట్లను విశ్లేషిస్తాయి
   - ముంచును సజీవం చేయడానికి `True`, ఈదని తీసివేయడానికి `False`ను రిటర్న్ చేయండి

3. **టూల్ సమీకరణ**
   - Python ఫంక్షన్లను AI టూల్స్ గా మార్చేందుకు `@ai_function` ఉపయోగించండి
   - అవసరమైనప్పుడు ఏజెంట్లు ఆటోమాటిగ్గా టూల్స్‌ను పిలిస్తాయి
   - టూల్స్ JSONని రిటర్న్ చేయడానికి ఏజెంట్లు విశ్లేషించవచ్చు

4. **నిర్మిత అవుట్పుట్లు**
   - టైపు-సురక్షిత డేటా తీయడానికి Pydantic మోడల్స్ ఉపయోగించండి
   - ఏజెంట్లు సృష్టించేటప్పుడు `response_format=MyModel` సెట్ చేయండి
   - `Model.model_validate_json()` తో ప్రతిస్పందనలను పరిష్కరించండి

5. **కస్టమ్ ఎగ్జిక్యూటర్స్**
   - వర్క్‌ఫ్లో భాగాలను సృష్టించడానికి `@executor(id="...")` ఉపయోగించండి
   - ఎగ్జిక్యూటర్స్ డేటాను మార్పిడి చేయవచ్చు లేదా సైడ్ ఎఫెక్ట్స్ నిర్వహించవచ్చు
   - వర్క్‌ఫ్లో ఫలితాలను ఉత్పత్తి చేయడానికి `ctx.yield_output()` ఉపయోగించండి

### 🚀 వాస్తవ ప్రపంచ అనువర్తనాలు:

- **యాత్ర బుకింగ్**: అందుబాటును తనిఖీ చేయండి, ప్రత్యామ్నాయాలు సూచించండి, ఎంపికలను పోల్చండి
- **కస్టమర్ సర్వీస్**: సమస్య రకం, భావోద్వేగం, ప్రాధాన్యత ఆధారంగా మార్గం
- **ఈ-కామర్స్**: ఇన్‌వెంటరీ తనిఖీ, ప్రత్యామ్నాయాలు సూచించండి, ఆర్డర్లను ప్రాసెస్ చేయండి
- **కంటెంట్ మోడరేషన్**: విషాక్తి స్కోర్లు, వినియోగదారుల ఫ్లాగ్‌ల ఆధారంగా మార్గం
- **ఆమోద వర్క్‌ఫ్లోలు**: మొత్తం, వినియోగదారు పాత్ర, ప్రమాద స్థాయి ఆధారంగా మార్గం
- **బహుస్థాయి ప్రాసెసింగ్**: డేటా నాణ్యత, సంపూర్ణత ఆధారంగా మార్గం

### 📚 తదుపరి దశలు:

- మరింత సంక్లిష్టమైన షరతులను జోడించండి (బహుళ ప్రమాణాలు)
- వర్క్‌ఫ్లో రాష్ట్ర నిర్వహణతో లూపులను అమలు చేయండి
- పునర్వినియోగపరచగల భాగాల కోసం సబ్-వర్క్‌ఫ్లోలను జోడించండి
- వాస్తవ APIలతో సమీకరణ చేయండి (హోటల్ బుకింగ్, ఇన్‌వెంటరీ వ్యవస్థలు)
- లోప నిర్వహణ మరియు ప్రత్యామ్నాయ మార్గాలను జోడించండి
- స్వయంగా ఉన్న విజువలైజేషన్ టూల్స్‌తో వర్క్‌ఫ్లోలను వీక్షించండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
